# Installations

In [1]:
#mount drive
from google.colab import drive
drive.mount('/content/drive')

#move to working directory
#%cd /content/drive/My Drive/Colab Notebooks/ #MSG
%cd /content/drive/MyDrive/Lucy Colab Notebooks

Mounted at /content/drive
/content/drive/MyDrive/Lucy Colab Notebooks


In [2]:
!pip install biopython
!pip install matplotlib-venn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 13.2 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import seaborn as sns
import scipy.stats as stats
import scipy.signal as signal
from scipy import cluster
from Bio import SeqIO
%matplotlib inline

sns.set_style('white')
plt.rcParams['xtick.labelsize']=15
plt.rcParams['ytick.labelsize']=15

# functions & data

In [45]:
### functions for loading ChIP data
def gaussian_smooth(data_array, index_step, sigma):
    mu = 0
    bins = np.arange(-4*sigma, 4*sigma, index_step, dtype=np.float32)
    gaussian = index_step*1/(sigma * np.sqrt(2 * np.pi))*np.exp( - (bins - mu)**2 / (2 * sigma**2) )
    return signal.convolve(data_array,gaussian,mode='same')


def initialize_rpm(data,index_step,length):

    smoothed_data = np.zeros([1, length], dtype=np.float32)
    smoothed_data[0][:] = gaussian_smooth(data[0],index_step,5*index_step)
    rpm = np.zeros([1, length])
    normalization_factor=float(sum(smoothed_data[0][:]))/float(1000000)
    print(normalization_factor)
    for i in range(0,length):
        rpm[0][i]=float(smoothed_data[0][i])/normalization_factor

    return rpm

def loading_chip_data(name):
    coordinate=np.loadtxt(name, dtype=np.int32, delimiter='\t', skiprows=2)
    chip_data=np.zeros([1,len(caulobacterfasta.seq)])
    ind=coordinate[:,0]<len(caulobacterfasta.seq)
    chip_data[0][coordinate[ind,0]-1]=coordinate[ind,1]
    chip_rpm=initialize_rpm(chip_data, 10, len(caulobacterfasta.seq))
    return chip_rpm

In [46]:
class Genome:
    def __init__(self, genome_list,genome_annotation, start, end, strand, length):
        self.name=genome_list #list with every gene name such as CCNA_00001
        self.annotation=genome_annotation # gene annotation if there is one, if none stores NA
        self.start=start # stores translational start position for each gene
        self.end=end #stores end position of each gene
        self.strand=strand # + or - strand (+1 or -1)
        self.length=length # length of gene

def loading_fasta_gbk(file_name,typeoffile):
    """reads either fasta or gbk files, file type needs to be given as 'fasta' or 'genbank' """
    loaded=SeqIO.read(file_name, typeoffile)
    return loaded


def reading_gbk_new(gbk_file, features_to_extract):
    """function that will load from the gbk file: the start, end, strand and length of gene as well as the name and annotated name/function.
    Returns one array and 2 lists """

    genome_gene=[]
    genome_gene_name=[]

    genome_start=[]
    genome_end=[]
    genome_strand=[]
    genome_length=[]
    print(len(features_to_extract))
    for i in range(0,len(gbk_file.features)):
        isfeature=False
        for j in range(len(features_to_extract)):
            if gbk_file.features[i].type == features_to_extract[j]:
                isfeature=True

        if isfeature==True:

            genome_gene.append(gbk_file.features[i].qualifiers['locus_tag'][0])

            if 'product' in gbk_file.features[i].qualifiers:
                genome_gene_name.append(gbk_file.features[i].qualifiers['product'][0])
            else:
                genome_gene_name.append('NA')

            if gbk_file.features[i].location.strand < 0 :
                genome_start.append(gbk_file.features[i].location.end)
                genome_end.append(gbk_file.features[i].location.start)
                genome_strand.append(-1)
                genome_length.append(abs(gbk_file.features[i].location.end-gbk_file.features[i].location.start)+1)
            else:
                genome_start.append(gbk_file.features[i].location.start)
                genome_end.append(gbk_file.features[i].location.end)
                genome_strand.append(1)
                genome_length.append(abs(gbk_file.features[i].location.end-gbk_file.features[i].location.start)+1)

    genome = Genome(genome_gene,genome_gene_name,genome_start,genome_end,genome_strand,genome_length)

    return genome

def load_rnaseq_data(name):
  """Loads RNA-seq data from a tab-delimited file into a pandas DataFrame."""
  rnaseq_data = pd.read_csv(name, sep='\t', index_col = None, skiprows=2, names=['position', 'value'])
  return rnaseq_data

In [47]:
caulobactergbk=loading_fasta_gbk('./data/NC_011916.gbk','genbank')
caulobacterfasta=loading_fasta_gbk('./data/NC_011916.fna','fasta')
genome=reading_gbk_new(caulobactergbk,['CDS','tRNA','rRNA','ncRNA'])
print(len(caulobactergbk))

4
4042929


In [48]:
##Genome representation
#puts a 1 where there is a gene

genome_gene_representation = np.zeros([1,len(caulobacterfasta.seq)])

for genes in [genome]:
    for i in range (0, len(genes.annotation)):
        if genes.strand[i]== +1:

            for j in range(0, genes.length[i]-1):
                genome_gene_representation[0][genes.start[i]+j]=10
        else:
            for j in range(0,genes.length[i]-1):
                genome_gene_representation[0][genes.end[i]+j]=-10

In [49]:
##@title loading ChIP data from GSE100657, Guo and Haakonsen 2018

#GapR - wild type
chip_356=loading_chip_data('./data/160614Lau_D16-5510_CcreNA1000_bestmap.sammacs.samout_treat_afterfiting_gi_221232939_ref_NC_011916.1_.wig')
#GapR-3xFLAG RFI treated
chip_rif=loading_chip_data('./data/160614Lau_D16-5513_CcreNA1000_bestmap.sammacs.samout_treat_afterfiting_gi_221232939_ref_NC_011916.1_.wig')
#NOTAG version
notag_chip=loading_chip_data('./data/160614Lau_D16-5512_CcreNA1000_bestmap.sammacs.samout_treat_afterfiting_gi_221232939_ref_NC_011916.1_.wig')

1999.826304
2000.08896
1998.060672


In [50]:
#making the diane data into a df so I can normalize it to the untagged version

diane = pd.DataFrame(notag_chip[0])
diane['356'] = chip_356[0]
diane['rif'] = chip_rif[0]
diane.columns = ['wt','gapR','gapR-rif']

diane.head()

,wt,gapR,gapR-rif
0,0.006930,0.005876,0.007479
1,0.009005,0.007557,0.009731
2,0.011425,0.009486,0.012384
3,0.014174,0.011640,0.015435
4,0.017218,0.013981,0.018867


In [51]:
#normalizing ChIP data by subtracting the untagged version

gapR_rif_norm = diane['gapR-rif'] - diane['wt']
gapR_wt_norm = diane['gapR'] - diane['wt']

In [52]:
# @title calculating ATcontent, caulos
def ATcontent_caulo(start, end):
    from Bio.Seq import Seq, MutableSeq
    # Calculate GC content
    sequence_slice = caulobacterfasta.seq[start:end]
    gc_count = sequence_slice.count('G') + sequence_slice.count('C')
    total_length = len(sequence_slice)
    if total_length == 0:
        return 0
    gc_content_percentage = (gc_count / total_length) * 100
    content = 100 - gc_content_percentage
    return content

### code a sliding window to record AT content
def sliding_window_caulo(window_length):
    sliding_array=np.zeros([1,len(caulobacterfasta.seq)])
    for i in range(int(window_length/2), len(caulobacterfasta.seq)-int(window_length/2)):
        start=i-int(window_length/2)
        end=i+int(window_length/2)
        sliding_array[0][i]=ATcontent_caulo(start, end)
    return sliding_array

sliding_300_caulo=sliding_window_caulo(300)

In [53]:
#Load data from GapR-3xFLAG RNA-seq (GSM2690554)
W356_RNArev_df = load_rnaseq_data('./data/D16-4815-1939G_align_out_norm_rv.wig')
W356_RNAfor_df = load_rnaseq_data('./data/D16-4815-1939G_align_out_norm_fw.wig')

In [54]:
#load TSStart and TSStop dataframes

# making the df_terminators variable
#excel_file_path = '/content/drive/MyDrive/Lucy Colab Notebooks/data/Caulo_TranscriptionStopSites.xlsx'
excel_file_path = './data/Caulo_TranscriptionStopSites.xlsx'
sheet_name = '4_terminators_Ccre'
df_terminators = pd.read_excel(excel_file_path, sheet_name=sheet_name, skiprows=7) # Skip the first 2 rows
df_terminators.columns = df_terminators.iloc[0] # Set the 3rd row (index 0 after skipping) as column headers
df_terminators = df_terminators[1:].reset_index(drop=True) # Remove the header row that was set as columns and reset index
#display(df_terminators)

# loading TSS dataframes
#tss_sites_df = pd.read_csv('/content/drive/MyDrive/Lucy Colab Notebooks/data/tss_sites_cleaned.csv')
tss_sites_df = pd.read_csv('./data/tss_sites_cleaned.csv')
# Display the head of each dataframe to verify loading
#print("TSS Sites DataFrame Head:")
#display(tss_sites_df.head())

#Generate top 5% peaks for WT and RIF ChIP, fill dataframe with ChIP info (from fig 8)




In [55]:
# @title making functions for supercoiling GapR enrichment

half_window = 100

# takes chip data and cutoff value, makes a list of lists that is above the cutoff value
#output of this is chipRegions input of getSequence

def enrichedRegions(chip_data, cutoff):
    x = chip_data[chip_data > cutoff]
    out = []
    z = x.index[0]
    start = x.index[0]
    for i in range(1,len(x.index)):
        if x.index[i] == z+1:
            z = x.index[i]
        else:
            end = z
            out.append([start,end])
            z = x.index[i]
            start = x.index[i]
    out.append([start,z])
    return out

def unenrichedRegions(chip_data, cutoff):
    x = chip_data[chip_data < cutoff]
    out = []
    z = x.index[0]
    start = x.index[0]
    for i in range(1,len(x.index)):
        if x.index[i] == z+1:
            z = x.index[i]
        else:
            end = z
            out.append([start,end])
            z = x.index[i]
            start = x.index[i]
    out.append([start,z])
    return out

def getuSequence(chipRegions, chip_data, genomeSequence, saveFile = False):
    chipMin = []
    sequence = []
    for i in chipRegions:
        if (i[1] - i[0]) > 100:
            cmin = chip_data.loc[i[0]:i[1]].idxmin()
            chipMin.append(cmin)
            sequence.append(str(genomeSequence.seq[cmin-half_window:cmin+half_window]))
    if saveFile != False:
        a = open(saveFile, 'w')
        for i in range(len(sequence)):
            a.write('>loc{}\n'.format(chipMin[i]))
            a.write(sequence[i] + '\n')
        a.close()

    return sequence


In [56]:
#find boundries for rif and wt ALL peaks
print(("bottom 5 percent rif"),(np.quantile(gapR_rif_norm, 0.05)))
print(("top 5 percent rif "), (np.quantile(gapR_rif_norm, 0.95)))
print(("bottom 5 percent wt"),(np.quantile(gapR_wt_norm, 0.05)))
print(("top 5 percent wt"), (np.quantile(gapR_wt_norm, 0.95)))

#get the ALL PEAK regions of rif and wt data
rif_enriched_windows_5 = enrichedRegions(gapR_rif_norm, 0.14768356894527906)
rif_nochip_windows_5 = unenrichedRegions(gapR_rif_norm, -0.09451845558257815)
wt_enriched_windows_5 = enrichedRegions(gapR_wt_norm, 0.19664495062840362)
wt_nochip_windows_5 = unenrichedRegions(gapR_wt_norm, -0.11562839831731775)

#turn the ALL PEAK regions of rif anf wt data into dfs for downstream processing
rif_enriched_windows_5 = pd.DataFrame(rif_enriched_windows_5, columns=['start', 'end'])
rif_nochip_windows_5 = pd.DataFrame(rif_nochip_windows_5, columns=['start', 'end'])
wt_enriched_windows_5 = pd.DataFrame(wt_enriched_windows_5, columns=['start', 'end'])
wt_nochip_windows_5 = pd.DataFrame(wt_nochip_windows_5, columns=['start', 'end'])

#calculate the number of peaks within each dataset
print(("_" * 30))
print(" " *30)
print ("number of top 5% GapR-ChIP peaks:" , len(wt_enriched_windows_5))
print ("number of top 5% GapR-ChIP valleys:" , len(wt_nochip_windows_5))
print ("number of top 5% Rif-GapR-ChIP peaks:" , len(rif_enriched_windows_5))
print ("number of top 5% Rif-GapR-ChIP valleys:" , len(rif_nochip_windows_5))

bottom 5 percent rif -0.09451845558257815
top 5 percent rif  0.14768356894527906
bottom 5 percent wt -0.11562839831731775
top 5 percent wt 0.19664495062840362
______________________________
                              
number of top 5% GapR-ChIP peaks: 692
number of top 5% GapR-ChIP valleys: 1602
number of top 5% Rif-GapR-ChIP peaks: 689
number of top 5% Rif-GapR-ChIP valleys: 1905


In [57]:
#find boundries for rif and wt ALL peaks
print(("bottom 20 percent rif"),(np.quantile(gapR_rif_norm, 0.2)))
print(("top 20 percent rif "), (np.quantile(gapR_rif_norm, 0.8)))
print(("bottom 20 percent wt"),(np.quantile(gapR_wt_norm, 0.2)))
print(("top 20 percent wt"), (np.quantile(gapR_wt_norm, 0.8)))

#get the ALL PEAK regions of rif and wt data
rif_enriched_windows_20 = enrichedRegions(gapR_rif_norm, 0.035502886446707856)
rif_nochip_windows_20 = unenrichedRegions(gapR_rif_norm, -0.0617211428049998)
wt_enriched_windows_20 = enrichedRegions(gapR_wt_norm,  0.04790532983120593)
wt_nochip_windows_20 = unenrichedRegions(gapR_wt_norm, -0.07835433846067265)

#turn the ALL PEAK regions of rif anf wt data into dfs for downstream processing
rif_enriched_windows_20 = pd.DataFrame(rif_enriched_windows_20, columns=['start', 'end'])
rif_nochip_windows_20 = pd.DataFrame(rif_nochip_windows_20, columns=['start', 'end'])
wt_enriched_windows_20 = pd.DataFrame(wt_enriched_windows_20, columns=['start', 'end'])
wt_nochip_windows_20 = pd.DataFrame(wt_nochip_windows_20, columns=['start', 'end'])

#calculate the number of peaks within each dataset
print(("_" * 30))
print(" " *30)
print ("number of top 20% GapR-ChIP peaks:" , len(wt_enriched_windows_20))
print ("number of top 20% GapR-ChIP valleys:" , len(wt_nochip_windows_20))
print ("number of top 20% Rif-GapR-ChIP peaks:" , len(rif_enriched_windows_20))
print ("number of top 20% Rif-GapR-ChIP valleys:" , len(rif_nochip_windows_20))

bottom 20 percent rif -0.0617211428049998
top 20 percent rif  0.035502886446707856
bottom 20 percent wt -0.07835433846067265
top 20 percent wt 0.04790532983120593
______________________________
                              
number of top 20% GapR-ChIP peaks: 2261
number of top 20% GapR-ChIP valleys: 3846
number of top 20% Rif-GapR-ChIP peaks: 2748
number of top 20% Rif-GapR-ChIP valleys: 4611


In [58]:
# Function to find the position of the maximum value within a window
def get_max_chip_position(row, chip_data):
    start = int(row['start'])
    end = int(row['end'])
    window_data = chip_data[start:end+1]
    max_index_in_window = np.argmax(window_data)

    return start + max_index_in_window


# Function to find the position of the minimum value within a window
def get_min_chip_position(row, chip_data):
    start = int(row['start'])
    end = int(row['end'])
    window_data = chip_data[start:end+1]
    min_index_in_window = np.argmin(window_data)

    return start + min_index_in_window


In [59]:
#also run on all peaks in the rif dataset
rif_enriched_windows_5['max_chip_position'] = rif_enriched_windows_5.apply(lambda row: get_max_chip_position(row, gapR_rif_norm), axis=1)
rif_nochip_windows_5['min_chip_position'] = rif_nochip_windows_5.apply(lambda row: get_min_chip_position(row, gapR_rif_norm), axis=1)
wt_enriched_windows_5['max_chip_position'] = wt_enriched_windows_5.apply(lambda row: get_max_chip_position(row, gapR_wt_norm), axis=1)
wt_nochip_windows_5['min_chip_position'] = wt_nochip_windows_5.apply(lambda row: get_min_chip_position(row, gapR_wt_norm), axis=1)

rif_enriched_windows_20['max_chip_position'] = rif_enriched_windows_20.apply(lambda row: get_max_chip_position(row, gapR_rif_norm), axis=1)
rif_nochip_windows_20['min_chip_position'] = rif_nochip_windows_20.apply(lambda row: get_min_chip_position(row, gapR_rif_norm), axis=1)
wt_enriched_windows_20['max_chip_position'] = wt_enriched_windows_20.apply(lambda row: get_max_chip_position(row, gapR_wt_norm), axis=1)
wt_nochip_windows_20['min_chip_position'] = wt_nochip_windows_20.apply(lambda row: get_min_chip_position(row, gapR_wt_norm), axis=1)


In [60]:
def get_chip_at_position(row, chip_data):
    position = int(row['max_chip_position'])
    chip = chip_data[position]
    return chip

In [61]:
#get max chIP in window for both wt and rif datasets
wt_enriched_windows_5['max_wt_chip_in_window'] = wt_enriched_windows_5.apply(lambda row: gapR_wt_norm.loc[int(row['max_chip_position'])], axis=1)
rif_enriched_windows_5['max_rif_chip_in_window'] = rif_enriched_windows_5.apply(lambda row: gapR_rif_norm.loc[int(row['max_chip_position'])], axis=1)

wt_enriched_windows_5['rif_chip_at_position'] = wt_enriched_windows_5.apply(lambda row: gapR_rif_norm.loc[int(row['max_chip_position'])], axis=1)
rif_enriched_windows_5['wt_chip_at_position'] = rif_enriched_windows_5.apply(lambda row: gapR_wt_norm.loc[int(row['max_chip_position'])], axis=1)

wt_enriched_windows_20['max_wt_chip_in_window'] = wt_enriched_windows_20.apply(lambda row: gapR_wt_norm.loc[int(row['max_chip_position'])], axis=1)
rif_enriched_windows_20['max_rif_chip_in_window'] = rif_enriched_windows_20.apply(lambda row: gapR_rif_norm.loc[int(row['max_chip_position'])], axis=1)

wt_enriched_windows_20['rif_chip_at_position'] = wt_enriched_windows_20.apply(lambda row: gapR_rif_norm.loc[int(row['max_chip_position'])], axis=1)
rif_enriched_windows_20['wt_chip_at_position'] = rif_enriched_windows_20.apply(lambda row: gapR_wt_norm.loc[int(row['max_chip_position'])], axis=1)


print("WT Enriched Windows 5 with Max Chip in Window:")
display(wt_enriched_windows_5.head())

print("\nRif Enriched Windows 5 with Max Chip in Window:")
display(rif_enriched_windows_5.head())

WT Enriched Windows 5 with Max Chip in Window:


,start,end,max_chip_position,max_wt_chip_in_window,rif_chip_at_position
0,5101,5593,5314,1.043464,1.274034
1,7961,8108,8022,0.245565,0.203822
2,11402,11456,11428,0.210334,0.024209
3,14416,14708,14594,0.260548,-0.003229
4,14960,15250,15034,0.245053,-0.007560



Rif Enriched Windows 5 with Max Chip in Window:


,start,end,max_chip_position,max_rif_chip_in_window,wt_chip_at_position
0,173,358,209,0.191218,-0.035344
1,1760,1830,1804,0.171878,-0.087655
2,2802,3011,2899,0.208343,-0.066884
3,4846,5728,5313,1.274365,1.043323
4,7967,8185,8102,0.248435,0.207249


calculate the difference in signals at the max chip peaks for WT and rif (used to calculate top 5%)

In [62]:
#calculate signal differencences
wt_enriched_windows_5['max_signal_difference'] = wt_enriched_windows_5['max_wt_chip_in_window'] - wt_enriched_windows_5['rif_chip_at_position']
wt_enriched_windows_20['max_signal_difference'] = wt_enriched_windows_20['max_wt_chip_in_window'] - wt_enriched_windows_20['rif_chip_at_position']


# calculate the fraction of the change that difference is
wt_enriched_windows_5['max_signal_frac_change'] = wt_enriched_windows_5['max_signal_difference'] / wt_enriched_windows_5['max_wt_chip_in_window']
wt_enriched_windows_20['max_signal_frac_change'] = wt_enriched_windows_20['max_signal_difference'] / wt_enriched_windows_20['max_wt_chip_in_window']


#grouping A/B and 'both' for meme analysis

In [63]:
#assign SUPERCOILING INDEPENDENT peaks (max chip signal does not decrease in response to rif)

def categorize_by_max_sign(df, threshold):
    group = []
    for _, row in df.iterrows():
        if row['max_signal_frac_change'] <= threshold: #this is the cutoff generated from top 5% change
            group.append('Group A')
        else:
            group.append('Group A/B')
    df['Group_by_max_sign'] = group
    return df

#only run the function on wt enriched windows
wt_enriched_windows_5 = categorize_by_max_sign(wt_enriched_windows_5.copy(), 0.05 )
wt_enriched_windows_20 = categorize_by_max_sign(wt_enriched_windows_20.copy(), 0.05 )


Group A is the supercoil-independent peaks, group B is all supercoil sensitive peaks, but can still include AT-dependent supercoil dependent binding. I generated two dataframes labled as such

In [64]:
groupA_5_df = wt_enriched_windows_5[wt_enriched_windows_5['Group_by_max_sign'] == 'Group A'].copy()
groupAB_5_df = wt_enriched_windows_5[wt_enriched_windows_5['Group_by_max_sign'] == 'Group A/B'].copy()

groupA_20_df = wt_enriched_windows_20[wt_enriched_windows_20['Group_by_max_sign'] == 'Group A'].copy()
groupAB_20_df = wt_enriched_windows_20[wt_enriched_windows_20['Group_by_max_sign'] == 'Group A/B'].copy()

Rif dataset has been shown to be AT-dependent in Figure 8, so the group B peaks that were shown to be supercoil-dependent (rif-sensitive) that are also in rif peak windows (AT-dependent) are considered BOTH AT/supercoil-dependent

In [65]:
#calculate if the peak form one class occurs in the window of another class:
def calculate_overlap_of_max_chip(df1, df2, max_chip_col_name, return_dataframe=False):

    num_overlaps = 0
    num_non_overlaps = 0
    is_overlapping_list = []

    # Ensure df2 'start' and 'end' columns are integer type for comparison
    df2_starts = df2['start'].astype(int).values
    df2_ends = df2['end'].astype(int).values

    for index, row in df1.iterrows():
        # Get the max chip position from df1 (as integer) using the provided column name
        max_chip_pos = int(row[max_chip_col_name])

        overlap_found = ((df2_starts <= max_chip_pos) & (max_chip_pos <= df2_ends)).any()  # creates a boolean Series. .any() checks if any value in this Series is True.
        is_overlapping_list.append(overlap_found)

        if overlap_found:
            num_overlaps += 1
        else:
            num_non_overlaps += 1

    if return_dataframe:
        df_out = df1.copy()
        df_out['is_overlapping'] = is_overlapping_list
        overlapping_df = df_out[df_out['is_overlapping']].drop(columns=['is_overlapping'])
        non_overlapping_df = df_out[~df_out['is_overlapping']].drop(columns=['is_overlapping'])
        return overlapping_df, non_overlapping_df
    else:
        return num_overlaps, num_non_overlaps


rif_overlapping_groupA_max_df, rif_non_overlapping_groupA_max_df = calculate_overlap_of_max_chip(groupA_5_df, rif_enriched_windows_5, 'max_chip_position', return_dataframe=True)
rif_overlapping_groupB_max_df, rif_non_overlapping_groupB_max_df = calculate_overlap_of_max_chip(groupAB_5_df, rif_enriched_windows_5, 'max_chip_position', return_dataframe=True)
rif_overlapping_groupA_max_df, rif_non_overlapping_groupA_max_df = calculate_overlap_of_max_chip(groupA_20_df, rif_enriched_windows_20, 'max_chip_position', return_dataframe=True)
rif_overlapping_groupB_max_df, rif_non_overlapping_groupB_max_df = calculate_overlap_of_max_chip(groupAB_20_df, rif_enriched_windows_20, 'max_chip_position', return_dataframe=True)

# Assign group B and group BOTH reflecting on if there is an overlap in rif
groupBOTH_5_df = rif_overlapping_groupB_max_df
groupB_5_df = rif_non_overlapping_groupB_max_df
groupBOTH_20_df = rif_overlapping_groupB_max_df
groupB_20_df = rif_non_overlapping_groupB_max_df


Now going to extract the DNA sequences from 50bp surrounding the peak for motif discovery

In [66]:
def extract_centered_sequences(df, genome_sequence, half_window_size=25, position_col='max_chip_position'):
    """
    Extracts DNA sequences of length (2 * half_window_size) centered around the
    specified position column
    """
    df_copy = df.copy()
    genome_length = len(genome_sequence.seq)
    extracted_sequences = []

    for _, row in df_copy.iterrows():
        center_pos = int(row[position_col])

        start = center_pos - half_window_size
        end = center_pos + half_window_size

        # Handle boundary conditions
        if start < 0:
            # If start is negative, extract from the end of the genome
            sequence_part_1 = genome_sequence.seq[genome_length + start:]
            sequence_part_2 = genome_sequence.seq[0:end]
            extracted_seq = str(sequence_part_1) + str(sequence_part_2)
        elif end > genome_length:
            # If end is beyond genome length, extract from the beginning of the genome
            sequence_part_1 = genome_sequence.seq[start:genome_length]
            sequence_part_2 = genome_sequence.seq[0:(end - genome_length)]
            extracted_seq = str(sequence_part_1) + str(sequence_part_2)
        else:
            extracted_seq = str(genome_sequence.seq[start:end])

        extracted_sequences.append(extracted_seq.upper())

    df_copy['centered_sequence_short'] = extracted_sequences
    return df_copy

# Apply the function to all specified DataFrames
groupA_5_df = extract_centered_sequences(groupA_5_df, caulobacterfasta)
groupAB_5_df = extract_centered_sequences(groupAB_5_df, caulobacterfasta)
groupBOTH_5_df = extract_centered_sequences(groupBOTH_5_df, caulobacterfasta)
groupB_5_df = extract_centered_sequences(groupB_5_df, caulobacterfasta)

groupA_20_df = extract_centered_sequences(groupA_20_df, caulobacterfasta)
groupAB_20_df = extract_centered_sequences(groupAB_20_df, caulobacterfasta)
groupBOTH_20_df = extract_centered_sequences(groupBOTH_20_df, caulobacterfasta)
groupB_20_df = extract_centered_sequences(groupB_20_df, caulobacterfasta)

rif_enriched_windows_5 = extract_centered_sequences(rif_enriched_windows_5, caulobacterfasta)
rif_enriched_windows_20 = extract_centered_sequences(rif_enriched_windows_20, caulobacterfasta)
wt_enriched_windows_5 = extract_centered_sequences(rif_enriched_windows_5, caulobacterfasta)
wt_enriched_windows_20 = extract_centered_sequences(rif_enriched_windows_20, caulobacterfasta)

# Display the head of groupA_5_df to show the new 'centered_sequence' column
display(groupA_5_df.head())

,start,end,max_chip_position,max_wt_chip_in_window,rif_chip_at_position,max_signal_difference,max_signal_frac_change,Group_by_max_sign,centered_sequence_short
0,5101,5593,5314,1.043464,1.274034,-0.230570,-0.220966,Group A,CGATCAGGGCAACCTGGTCGGCTTTTGCTTTGAATCCAACGCTCTA...
34,97742,97790,97755,0.213335,0.214352,-0.001017,-0.004768,Group A,CCGGCATTTGTAAAATTTCTCCAGATACGCGATCGAGATTGCGCGC...
45,170527,170911,170669,0.573838,0.628507,-0.054669,-0.095269,Group A,TCGCCCTGAAAGCGAATTCGTTCAGAACTCTGGCGCCGTCGTTGGA...
53,220808,220926,220856,0.249074,0.307315,-0.058242,-0.233833,Group A,CTTCGGGAAAATAATTCATCAAGGGTTGAATTTATTCGCGGCTCGG...
54,224847,225145,224980,0.662463,0.674395,-0.011931,-0.018011,Group A,CTGACCACTTCTTCCGCGTTTCCAGGGGTTTGTGACACTTTGTGGG...


In [67]:
import pandas as pd
from google.colab import files

def dataframe_to_fasta(df, seq_column, output_filename, header_prefix):
    """
    Convert a dataframe column of sequences into a FASTA-formatted text file.
    """

    with open(output_filename, "w") as f:
        for i, seq in enumerate(df[seq_column]):

            # Skip missing values
            if pd.isna(seq):
                continue

            seq = str(seq).strip().upper()

            # Write FASTA header
            f.write(f">{header_prefix}_{i}\n")

            # Wrap sequence at 60 characters per line (FASTA convention)
            for j in range(0, len(seq), 60):
                f.write(seq[j:j+60] + "\n")



# Run separately for each dataframe to create the FASTA files
dataframe_to_fasta(wt_enriched_windows_5, "centered_sequence_short", "wt_enriched_windows_5_short.txt", "wt_5")
dataframe_to_fasta(wt_enriched_windows_20, "centered_sequence_short", "wt_enriched_windows_20_short.txt", "wt_20")
dataframe_to_fasta(rif_enriched_windows_5, "centered_sequence_short", "rif_enriched_windows_5_short.txt", "rif_5")
dataframe_to_fasta(rif_enriched_windows_20, "centered_sequence_short", "rif_enriched_windows_20_short.txt", "rif_20")

dataframe_to_fasta(groupA_5_df, "centered_sequence_short", "groupA_5_short.txt", "groupA_5")
dataframe_to_fasta(groupA_20_df, "centered_sequence_short", "groupA_20_short.txt", "groupA_20")
dataframe_to_fasta(groupB_5_df, "centered_sequence_short", "groupB_5_short.txt", "groupB_5")
dataframe_to_fasta(groupB_20_df, "centered_sequence_short", "groupB_20_short.txt", "groupB_20")
dataframe_to_fasta(groupBOTH_5_df, "centered_sequence_short", "groupBOTH_5_short.txt", "groupBOTH_5")
dataframe_to_fasta(groupBOTH_20_df, "centered_sequence_short", "groupBOTH_20_short.txt", "groupBOTH_20")

# Now, download the generated files
files_to_download = [
    "wt_enriched_windows_5_short.txt",
    "wt_enriched_windows_20_short.txt",
    "rif_enriched_windows_5_short.txt",
    "rif_enriched_windows_20_short.txt",
    "groupA_5_short.txt",
    "groupA_20_short.txt",
    "groupB_5_short.txt",
    "groupB_20_short.txt",
    "groupBOTH_5_short.txt",
    "groupBOTH_20_short.txt"
]

for filename in files_to_download:
    try:
        files.download(filename)
        print(f"Downloaded: {filename}")
    except Exception as e:
        print(f"Error downloading {filename}: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: wt_enriched_windows_5_short.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloaded: groupBOTH_5_short.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: groupBOTH_20_short.txt


In [68]:
def generate_markov_model_background(sequence_obj, pseudocount=1, order=1):
    nucleotides = ['A', 'C', 'G', 'T']
    seq_str = str(sequence_obj.seq).upper()
    total_length = len(seq_str)

    # Initialize 0-order counts
    monomer_counts = {n: 0 for n in nucleotides}
    # Initialize 1st-order counts (dimers)
    dimer_counts = {n1 + n2: 0 for n1 in nucleotides for n2 in nucleotides}

    # Count occurrences
    for i in range(total_length):
        if seq_str[i] in nucleotides:
            monomer_counts[seq_str[i]] += 1
        if i < total_length - 1:
            dimer = seq_str[i] + seq_str[i+1]
            if all(n in nucleotides for n in dimer):
                dimer_counts[dimer] += 1

    # Apply pseudocounts
    for n in nucleotides:
        monomer_counts[n] += pseudocount
    for dim in dimer_counts:
        dimer_counts[dim] += pseudocount

    # Calculate 0-order probabilities
    total_monomer_count = sum(monomer_counts.values())
    monomer_probs = {n: count / total_monomer_count for n, count in monomer_counts.items()}

    # Calculate 1st-order conditional probabilities
    # P(N2|N1) = count(N1N2) / count(N1)
    conditional_dimer_probs = {}
    for n1 in nucleotides:
        # Sum of pseudocount-adjusted counts for all dimers starting with n1
        sum_counts_for_n1 = sum(dimer_counts[n1 + n2] for n2 in nucleotides)
        for n2 in nucleotides:
            dimer = n1 + n2
            conditional_dimer_probs[dimer] = dimer_counts[dimer] / sum_counts_for_n1

    # Format output string
    output_lines = []

    # 0-order model
    output_lines.append('# 0-order background model')
    output_lines.append(f'# order {0}')
    for n in nucleotides:
        output_lines.append(f'{n} {monomer_probs[n]:.6f}')
    output_lines.append(' ')

    # 1st-order model
    if order >= 1:
        output_lines.append('# 1st-order background model')
        output_lines.append(f'# order {1}')
        for n1 in nucleotides:
            for n2 in nucleotides:
                dimer = n1 + n2
                output_lines.append(f'{dimer} {conditional_dimer_probs[dimer]:.6f}')

    return '\n'.join(output_lines)

# Generate the Markov model background for caulobacterfasta with the corrected 1st-order format
# (uses pseudocount=0.125 and order=1)
markov_model_output = generate_markov_model_background(caulobacterfasta, pseudocount=0.125, order=1)

output_filename = 'caulobacter_background.txt'
with open(output_filename, 'w') as f:
    f.write(markov_model_output)
print(f"Saved generated Markov Model to {output_filename}")

# Display
with open(output_filename, 'r') as f:
    print(f.read())

Saved generated Markov Model to caulobacter_background.txt
# 0-order background model
# order 0
A 0.164715
C 0.336378
G 0.335338
T 0.163568
 
# 1st-order background model
# order 1
AA 0.179565
AC 0.287658
AG 0.322010
AT 0.210767
CA 0.166258
CC 0.287007
CG 0.390244
CT 0.156491
GA 0.199753
GC 0.373884
GG 0.286488
GT 0.139874
TA 0.074758
TC 0.410080
TG 0.335993
TT 0.179169


How to actually do the MEME analysis

there is a MEME Suite that you have to download and install
https://meme-suite.org/meme/


PARAMETERS USED FOR WEB EXTENSION VERSION OF MEME-motif discovery
- Classic mode
- DNA
- upload background model:
    - required ebcause CCres is so GC rich and it assumes == nucleotide occurance if you dont do that.
    - Generation of background markov model below.
    - minimum width: 6 maximum width: 25
- zero or one occurance per sequence
- number of motifs (3)
- look for palindromes only? (no)
- Should MEME randomize sequence order? (yes) I would hit no If my .txt file was through order of confidence. It is not



In [69]:
def combine_sequences(df, sequence_column='centered_sequence_short'):
    """
    Combines all sequences from a specified column in a DataFrame into a single string that can be used to measure AT percentage
    """
    combined_string = "".join(df[sequence_column].astype(str))
    return combined_string



In [70]:
# Execute the combine_sequences function on groupA_20_df
all_groupA_20_sequences = combine_sequences(groupA_20_df, 'centered_sequence_short')
all_groupB_20_sequences = combine_sequences(groupB_20_df, 'centered_sequence_short')
all_groupBOTH_20_sequences = combine_sequences(groupBOTH_20_df, 'centered_sequence_short')

# Print the first 100 characters to make sure it worked
print(f"First 100 characters of combined Group A 20% sequences: {all_groupA_20_sequences[:100]}")
print(f"First 100 characters of combined Group B 20% sequences: {all_groupB_20_sequences[:100]}")
print(f"First 100 characters of combined Group BOTH 20% sequences: {all_groupBOTH_20_sequences[:100]}")


First 100 characters of combined Group A 20% sequences: CGATCAGGGCAACCTGGTCGGCTTTTGCTTTGAATCCAACGCTCTATTTCCCTCCATAAGGAAGCCGTTTCCCCGACCCTTCGCTCACGAGCCCGGCCGA
First 100 characters of combined Group B 20% sequences: TCGTGTAATTCCTTAACGCCGCCCCTTAACGGGAGTCGCGACGCTCTGTACCGCTGAAGATCCGCCACAACGGCGTGCTGGGCTATTTCGTCGAGGCGAC
First 100 characters of combined Group BOTH 20% sequences: AACTTCGCACAACGCATTCCGCCCGTTTGATTTTCGTCGACGGTGCGGAGCCAGACGCTGACAATGCGCGACTATTACGAAATTCTCGGCGTGACCCGGA


Did the MEME motif analysis as outlined, and not going to download all of the sequences used to make each motif to calculate AT-content

In [71]:
path_to_motifs = "/content/drive/MyDrive/Lucy Colab Notebooks/MEME/all-motifs-MEME.xlsx"

In [72]:
MEME_motifs = pd.read_excel(path_to_motifs)
MEME_motifs.head()

,BL MOTIF GWHSDBSDHGWTTTY width=15 seqs=410,BL MOTIF HGGCGDYGNYGDHSD width=15 seqs=523,BL MOTIF TWAAWWCAAANDNTT width=15 seqs=209,BL MOTIF GAAAWH width=6 seqs=346
0,groupA_20_429 ( 21) GATGTTCTTGAAT...,groupB_20_89 ( 24) TGGCGATGATGTT...,groupBOTH_20_112 ( 18) TAAAATCAAATAT...,groupBOTH_20_735 ( 26) GAAAAA 1
1,groupA_20_106 ( 21) TAAGTGTTTGATT...,groupB_20_886 ( 6) TGGTGAAGATGAT...,groupBOTH_20_593 ( 18) TAAAATCAAACAT...,groupBOTH_20_725 ( 19) GAAAAA 1
2,groupA_20_277 ( 3) TAAGTATTTGATT...,groupB_20_895 ( 8) CGCCGGTGATGAT...,groupBOTH_20_289 ( 14) TAAAATCAAACAT...,groupBOTH_20_701 ( 19) GAAAAA 1
3,groupA_20_181 ( 14) TATCTATTTGTTT...,groupB_20_396 ( 19) TGGCGGCGATGAT...,groupBOTH_20_582 ( 15) TAAAATCATATAT...,groupBOTH_20_680 ( 36) GAAAAA 1
4,groupA_20_37 ( 34) GTTGATCTGGATT...,groupB_20_227 ( 31) AGGCCTCGATGTT...,groupBOTH_20_353 ( 27) TAAAATCAAACAC...,groupBOTH_20_666 ( 27) GAAAAA 1


In [73]:
print(MEME_motifs['BL   MOTIF GAAAWH width=6 seqs=346'])

0      groupBOTH_20_735         (   26) GAAAAA  1 
1      groupBOTH_20_725         (   19) GAAAAA  1 
2      groupBOTH_20_701         (   19) GAAAAA  1 
3      groupBOTH_20_680         (   36) GAAAAA  1 
4      groupBOTH_20_666         (   27) GAAAAA  1 
                          ...                     
518                                            NaN
519                                            NaN
520                                            NaN
521                                            NaN
522                                            NaN
Name: BL   MOTIF GAAAWH width=6 seqs=346, Length: 523, dtype: object


In [74]:
new_df = pd.DataFrame()

In [75]:
from types import new_class
def clean_motifs(df, new_df, column_name, new_column_name):
    """
    Cleans info that isnt the sequence from each row

    """

    for index, row in df.iterrows():
        sequence = str(row[column_name])[33:48]
        new_df.loc[index, new_column_name] = sequence

new_df.head()

""


In [76]:
clean_motifs(MEME_motifs, new_df, 'BL   MOTIF GWHSDBSDHGWTTTY width=15 seqs=410', 'AT_dependent')
clean_motifs(MEME_motifs, new_df, 'BL   MOTIF HGGCGDYGNYGDHSD width=15 seqs=523', 'SC_dependent')
clean_motifs(MEME_motifs, new_df, 'BL   MOTIF TWAAWWCAAANDNTT width=15 seqs=209', 'BOTH_dependent_1')
new_df.head()

,AT_dependent,SC_dependent,BOTH_dependent_1
0,GATGTTCTTGAATTT,TGGCGATGATGTTCT,TAAAATCAAATATTT
1,TAAGTGTTTGATTTT,TGGTGAAGATGATGA,TAAAATCAAACATTT
2,TAAGTATTTGATTTT,CGCCGGTGATGATCA,TAAAATCAAACATTT
3,TATCTATTTGTTTTT,TGGCGGCGATGATCT,TAAAATCATATATTT
4,GTTGATCTGGATTTT,AGGCCTCGATGTTCA,TAAAATCAAACACTT


In [77]:
def clean_motifs2(df, new_df, column_name, new_column_name):
    """
    Cleans info that isnt the sequence from each row

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column containing the sequences.

    Returns:
        df (pd.DataFrame): The cleaned DataFrame.
    """

    for index, row in df.iterrows():
        sequence = str(row[column_name])[33:39]
        #print(sequence)
        new_df.loc[index, new_column_name] = sequence

new_df.head()

,AT_dependent,SC_dependent,BOTH_dependent_1
0,GATGTTCTTGAATTT,TGGCGATGATGTTCT,TAAAATCAAATATTT
1,TAAGTGTTTGATTTT,TGGTGAAGATGATGA,TAAAATCAAACATTT
2,TAAGTATTTGATTTT,CGCCGGTGATGATCA,TAAAATCAAACATTT
3,TATCTATTTGTTTTT,TGGCGGCGATGATCT,TAAAATCATATATTT
4,GTTGATCTGGATTTT,AGGCCTCGATGTTCA,TAAAATCAAACACTT


In [78]:
clean_motifs2(MEME_motifs, new_df, 'BL   MOTIF GAAAWH width=6 seqs=346', 'BOTH_dependent_2')
new_df.head()

,AT_dependent,SC_dependent,BOTH_dependent_1,BOTH_dependent_2
0,GATGTTCTTGAATTT,TGGCGATGATGTTCT,TAAAATCAAATATTT,GAAAAA
1,TAAGTGTTTGATTTT,TGGTGAAGATGATGA,TAAAATCAAACATTT,GAAAAA
2,TAAGTATTTGATTTT,CGCCGGTGATGATCA,TAAAATCAAACATTT,GAAAAA
3,TATCTATTTGTTTTT,TGGCGGCGATGATCT,TAAAATCATATATTT,GAAAAA
4,GTTGATCTGGATTTT,AGGCCTCGATGTTCA,TAAAATCAAACACTT,GAAAAA


In [79]:
# Execute the combine_sequences function on new_df, using the columns created by clean_motifs
#this can be used to claculate AT%
all_groupA_20_sequences = combine_sequences(new_df, 'AT_dependent')
all_groupB_20_sequences = combine_sequences(new_df, 'SC_dependent')
all_groupBOTH_20_1_sequences = combine_sequences(new_df, 'BOTH_dependent_1')
all_groupBOTH_20_2_sequences = combine_sequences(new_df, 'BOTH_dependent_2')

# Print the first 100 characters
print(f"First 100 characters of combined Group A 20% sequences: {all_groupA_20_sequences[:100]}")
print(f"First 100 characters of combined Group B 20% sequences: {all_groupB_20_sequences[:100]}")
print(f"First 100 characters of combined Group BOTH 20% sequences: {all_groupBOTH_20_1_sequences[:100]}")
print(f"First 100 characters of combined Group BOTH 20% sequences: {all_groupBOTH_20_2_sequences[:100]}")

First 100 characters of combined Group A 20% sequences: GATGTTCTTGAATTTTAAGTGTTTGATTTTTAAGTATTTGATTTTTATCTATTTGTTTTTGTTGATCTGGATTTTGTCGTTGAAGTTTTCTTATTTGAAG
First 100 characters of combined Group B 20% sequences: TGGCGATGATGTTCTTGGTGAAGATGATGACGCCGGTGATGATCATGGCGGCGATGATCTAGGCCTCGATGTTCAAGGCGATGTTGTTCGTGGCGATGAT
First 100 characters of combined Group BOTH 20% sequences: TAAAATCAAATATTTTAAAATCAAACATTTTAAAATCAAACATTTTAAAATCATATATTTTAAAATCAAACACTTTAAAATCAAACACTTTAAAATCAAA
First 100 characters of combined Group BOTH 20% sequences: GAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAAAAGAAA
